# Cycle 1 — Data Exploration: `skysports_match_stats.csv`

**Project:** Football Predictor — Match Outcome Prediction (Win / Draw / Loss)  
**Dataset:** `skysports_match_stats.csv`  
**Source:** Scraped from Sky Sports match pages. Contains detailed in-match statistics per game.  
**Coverage:** Premier League seasons approximately 2020/21 to 2022/23 (1,140 matches)

---

## Purpose of this Notebook

This notebook performs **exploratory data analysis (EDA)** on the secondary dataset. The goal is to:
- Understand its structure, columns, and data types
- Compare it against `premier_league_matches.csv`
- Identify data quality issues and data leakage risks
- Determine what can and cannot be used for pre-match prediction
- Document all findings for preprocessing and the final report

---
## Key Findings at a Glance

These are the most important things to know about this dataset before doing anything with it.

### 1. Critical data leakage — all match statistics are post-match
Every statistic in this dataset — possession, shots, tackles, saves, corners, fouls, cards — is recorded **after the match has been played**. You cannot know these values before kickoff. This means none of them can be used directly as prediction features. If you trained a model on these raw values, it would be cheating: the model would learn patterns from information that does not exist at prediction time, producing artificially high accuracy that would fail completely in real use. The solution is to engineer **rolling averages** from a team's past matches — for example, their average possession over the last 5 games. These are known before the match starts and are therefore valid features.

### 2. `Goals Home` and `Away Goals` directly reveal the result
These two columns are the full-time scoreline. If home goals > away goals, the result is a Home Win. They must be dropped before any model training — keeping them would be the most severe form of data leakage possible. The actual target variable is the `class` column (see Finding 3).

### 3. The target variable is correct
Unlike `premier_league_matches.csv` where `FTR` was broken (binary H/NH), this dataset has a proper 3-class target in the `class` column: `h` (Home Win), `d` (Draw), `a` (Away Win). This is exactly what our model needs to predict. It will be encoded as H=2, D=1, A=0 in preprocessing.

### 4. Only 3 seasons of data (1,140 rows)
This dataset covers approximately 2020/21 to 2022/23 — just 3 Premier League seasons. Compared to `premier_league_matches.csv` which has 6,840 rows across 20 seasons, this is significantly less data. Less data generally means the model has fewer examples to learn from, which can hurt accuracy. However, the richness of the statistics (possession, shots, tackles, etc.) may compensate if rolling features are engineered correctly. This trade-off will be evaluated in the modelling phase.

### 5. No missing values
All 40 columns are complete across all 1,140 rows. No imputation needed.

### 6. Draws are underrepresented
Home wins: 494 (43.3%), Away wins: 390 (34.2%), Draws: 256 (22.5%). This class imbalance means the model will naturally find draws harder to predict correctly. Balanced class weights should be used during training to compensate.

---

---
## Cell 1 — Load the Dataset

**What it does:** Loads `skysports_match_stats.csv` into a DataFrame, prints the shape, and displays the first 5 rows.

**Why:** First step to understand the basic dimensions and structure of the dataset.

In [ ]:
import pandas as pd

df = pd.read_csv('../data/raw/skysports_match_stats.csv')
print(df.shape)
df.head()

### Output
- Shape: **(1140, 40)** — 1,140 matches, 40 columns
- First rows show matches from 28th May 2023

### Observations
- Significantly smaller than `premier_league_matches.csv` (1,140 vs 6,840 rows)
- Only covers recent seasons (~2020/21 to 2022/23)
- Contains rich in-match statistics: possession, shots, tackles, passes, corners, fouls, cards
- `Home Team` and `Away Team` are already encoded as integers (1–25) — team names are not present
- `date` is in a non-standard format: `28th May 2023`
- `attendance` contains commas: `60,095` — stored as a string, not a number

### Issues to Flag
- Fewer rows means less training data if used alone
- Non-standard date format needs parsing in preprocessing
- Attendance needs cleaning (remove commas, convert to integer)

---
## Cell 2 — Column Names

**What it does:** Prints all 40 column names as a list.

**Why:** Ensures we can see every column — `df.head()` truncates with `...`.

In [ ]:
print(df.columns.tolist())

### Output
```
['date', 'clock', 'stadium', 'class', 'attendance', 'Home Team', 'Goals Home',
 'Away Team', 'Away Goals', 'home_possessions', 'away_possessions',
 'home_shots', 'away_shots', 'home_on', 'away_on', 'home_off', 'away_off',
 'home_blocked', 'away_blocked', 'home_pass', 'away_pass',
 'home_chances', 'away_chances', 'home_corners', 'away_corners',
 'home_offside', 'away_offside', 'home_tackles', 'away_tackles',
 'home_duels', 'away_duels', 'home_saves', 'away_saves',
 'home_fouls', 'away_fouls', 'home_yellow', 'away_yellow',
 'home_red', 'away_red', 'links']
```

### Column Reference Guide

| Column | Description | Usable for Prediction? |
|---|---|---|
| `date` | Match date (non-standard format) | No — drop after feature engineering |
| `clock` | Kick-off time | No — not predictive |
| `stadium` | Venue name | Possibly — encodes home/away advantage |
| `class` | Match result: h/d/a — **TARGET VARIABLE** | Yes — this is what we predict |
| `attendance` | Crowd size (string with commas) | Possibly — but noisy and often correlated with team |
| `Home Team` | Home team (encoded as integer 1–25) | Yes |
| `Goals Home` | Goals scored by home team | **NO — DATA LEAKAGE** |
| `Away Team` | Away team (encoded as integer 1–25) | Yes |
| `Away Goals` | Goals scored by away team | **NO — DATA LEAKAGE** |
| `home_possessions` | Home team possession % | **NO — post-match stat** |
| `away_possessions` | Away team possession % | **NO — post-match stat** |
| `home_shots` / `away_shots` | Total shots | **NO — post-match stat** |
| `home_on` / `away_on` | Shots on target | **NO — post-match stat** |
| `home_off` / `away_off` | Shots off target | **NO — post-match stat** |
| `home_blocked` / `away_blocked` | Blocked shots | **NO — post-match stat** |
| `home_pass` / `away_pass` | Pass accuracy % | **NO — post-match stat** |
| `home_chances` / `away_chances` | Big chances created | **NO — post-match stat** |
| `home_corners` / `away_corners` | Corners | **NO — post-match stat** |
| `home_offside` / `away_offside` | Offsides | **NO — post-match stat** |
| `home_tackles` / `away_tackles` | Tackle success % | **NO — post-match stat** |
| `home_duels` / `away_duels` | Duel win % | **NO — post-match stat** |
| `home_saves` / `away_saves` | Goalkeeper saves | **NO — post-match stat** |
| `home_fouls` / `away_fouls` | Fouls committed | **NO — post-match stat** |
| `home_yellow` / `away_yellow` | Yellow cards | **NO — post-match stat** |
| `home_red` / `away_red` | Red cards | **NO — post-match stat** |
| `links` | URL to Sky Sports match page | No — drop, not a feature |

### Observations
- `class` is the target variable — contains `h`, `d`, `a` (correct 3-class labels)
- `Goals Home` and `Away Goals` directly reveal the result — critical leakage
- **Every match statistic in this dataset is a post-match value** — you cannot know possession, shots, tackles, or saves before the match ends
- `links` is a web URL — not a predictive feature, must be dropped

---
## Cell 3 — Data Types

**What it does:** Prints the data type of every column.

**Why:** Identifies which columns need type conversion or encoding before modelling.

In [ ]:
print(df.dtypes)

### Output
```
date                object
clock               object
stadium             object
class               object
attendance          object
Home Team            int64
Goals Home           int64
Away Team            int64
Away Goals           int64
home_possessions   float64
away_possessions   float64
home_shots           int64
away_shots           int64
... (remaining stats are int64 or float64)
links               object
```

### Observations
- `Home Team` and `Away Team` are already `int64` — team names have been pre-encoded as numbers (1–25)
- `attendance` is `object` (string) despite being a number — caused by the comma formatting (e.g. `60,095`)
- `home_tackles`, `away_tackles`, `home_duels`, `away_duels`, `home_pass`, `away_pass` are `float64` — these are percentage values (e.g. 82.4%)
- `class` (target) is `object` — needs encoding to numeric for modelling

### Notes for Preprocessing
- `attendance`: strip commas, convert to `int64`
- `class`: encode as H=2, D=1, A=0 (consistent with `premier_league_matches.csv` encoding)
- `date`: parse to datetime for sorting and feature engineering
- Drop: `clock`, `stadium`, `links`

---
## Cell 4 — Target Variable Distribution

**What it does:** Counts how many times each value appears in the `class` column (the target variable).

**Why:** We need to verify the target contains 3 correct classes and check for class imbalance.

In [ ]:
print("Target variable distribution:")
print(df['class'].value_counts())

### Output
```
class
h    494
a    390
d    256
```

### Observations
- 3 correct classes present: `h` (Home Win), `a` (Away Win), `d` (Draw)
- Unlike `premier_league_matches.csv`, **this target is not broken** — no binary conversion issue
- **Class imbalance exists:**
  - Home Win: 494 (43.3%)
  - Away Win: 390 (34.2%)
  - Draw: 256 (22.5%)
- Draws are underrepresented — this mirrors real football where draws are the least common outcome
- Home wins remain the most frequent — consistent with home advantage observed in `premier_league_matches.csv`

### Notes for Preprocessing & Modelling
- Class imbalance means the model may struggle to predict draws correctly
- Consider using `class_weight='balanced'` in Logistic Regression and Random Forest
- XGBoost handles imbalance better natively via `scale_pos_weight`
- For the report: draw prediction is a known challenge in football ML — worth mentioning

---
## Cell 5 — Missing Values

**What it does:** Counts null/missing values in every column.

**Why:** Missing values must be handled before training any ML model.

In [ ]:
print("Missing values:")
print(df.isnull().sum())

### Output
All 40 columns show **0 missing values**.

### Observations
- The dataset is complete — no missing value handling required
- This is consistent with `premier_league_matches.csv` — both datasets are clean in this regard

### Notes
- No imputation needed in preprocessing

---
## Cell 6 — CRITICAL: Data Leakage Check

**What it does:** Inspects `Goals Home` and `Away Goals` to confirm they are post-match values that leak the result.

**Why:** Data leakage is when information that would not be available at prediction time is included as a feature. This causes artificially high accuracy that would not hold in real use.

In [ ]:
print("Goals Home distribution:")
print(df['Goals Home'].value_counts().sort_index())
print()
print("Away Goals distribution:")
print(df['Away Goals'].value_counts().sort_index())

### Output
```
Goals Home distribution:
0    285
1    374
2    254
3    135
4     58
...

Away Goals distribution:
0    356
1    370
2    230
3    120
4     44
...
```

### CRITICAL ISSUE — Data Leakage

`Goals Home` and `Away Goals` are the **full-time goals scored** in the match. They directly determine the result:
- `Goals Home > Away Goals` → Home Win
- `Goals Home < Away Goals` → Away Win
- `Goals Home == Away Goals` → Draw

Including these as features would give the model the answer before it predicts — this is **data leakage**. A model trained with these features would achieve near-perfect accuracy, but would be completely useless in practice (you cannot know the goals before the match is played).

The same applies to **all other match statistics** in this dataset (possession, shots, tackles, saves, etc.) — they are all measured **during or after the match**.

### How to Use This Dataset Correctly
To use `skysports_match_stats.csv` for **pre-match prediction**, we must:
1. **Drop all current-match statistics** as direct features
2. **Engineer rolling averages** from past matches — e.g. a team's average possession over their last 5 games
3. These rolling averages are known **before** the match and are therefore valid features

### Notes for Report
- This is a key methodological decision: the difference between post-match analysis and pre-match prediction
- Using raw match stats without rolling aggregation would invalidate the entire model
- This is the core reason feature engineering is required for this dataset

---
## Cell 7 — Possession Check

**What it does:** Verifies that `home_possessions` and `away_possessions` sum to approximately 100% per match.

**Why:** A sanity check to confirm the possession values are valid percentages.

In [ ]:
possession_sum = df['home_possessions'] + df['away_possessions']
print("Possession sum stats:")
print(possession_sum.describe())

### Output
```
count    1140.000000
mean      100.022719
std         0.896497
min        96.000000
max       130.000000
```

### Observations
- Mean is ~100 — possession values are valid percentages in most rows
- A small number of rows have a sum above 100 (max 130) — this is a data quality issue from scraping
- The standard deviation of 0.90 suggests most values are very close to 100

### Notes for Preprocessing
- Flag and inspect rows where possession sum deviates significantly from 100
- Consider capping or correcting obvious scraping errors
- When engineering rolling possession features, use the raw values but be aware of this noise

---
## Cell 8 — Summary Statistics

**What it does:** Produces descriptive statistics for all numeric columns.

**Why:** Reveals the range, scale, and distribution of each feature — useful for spotting outliers and understanding what rolling averages might look like.

In [ ]:
df.describe()

### Key Observations

**Goals:**
- Average home goals: **1.50**, average away goals: **1.29**
- Home advantage confirmed again — consistent with `premier_league_matches.csv`
- Max home goals: **9**, max away goals: **7** — same extremes as the other dataset

**Possession (`home_possessions`, `away_possessions`):**
- Home team averages **50.8%**, away team **49.2%** — home teams tend to dominate possession slightly
- Range: 17.9% to 82.4% — wide spread, possession varies greatly by match

**Shots (`home_shots`, `away_shots`):**
- Home team averages **13.6 shots**, away team **11.5 shots** per game
- Home teams generate more shots — another sign of home advantage

**Tackles (`home_tackles`, `away_tackles`):**
- These are stored as **tackle success percentages** (float), not raw counts
- Average ~57-58% success rate for both teams

**Cards:**
- Yellow cards: average ~1.6 per team per game
- Red cards: average ~0.05 — very rare, as expected

**Team encoding (`Home Team`, `Away Team`):**
- Values range from 1 to 25 — 25 unique teams across all seasons
- The encoding is already done but the mapping (which number = which team) is not documented in the dataset

### Notes for Preprocessing
- Features have very different scales (e.g. `home_pass` is 49–94%, `home_shots` is 1–33)
- StandardScaler will be needed for Logistic Regression
- The team encoding (1–25) is anonymous — we don't know which number maps to which team without the original scraper code

---
## Cell 9 — Date Range

**What it does:** Checks the earliest and latest dates in the dataset to understand the time coverage.

**Why:** Knowing the date range tells us which seasons are covered and helps us understand how many rolling features we can engineer.

In [ ]:
print("First match (earliest):", df['date'].iloc[-1])
print("Last match (latest):", df['date'].iloc[0])

### Output
```
First match (earliest): 12/9/2020
Last match (latest): 28th May 2023
```

### Observations
- Covers approximately **3 Premier League seasons**: 2020/21, 2021/22, 2022/23
- Dates are stored in two different formats: `12/9/2020` and `28th May 2023` — inconsistent formatting from scraping
- Only 3 seasons means significantly less historical context than `premier_league_matches.csv` (20+ seasons)

### Notes for Preprocessing
- Parse dates carefully — handle both formats (`dd/mm/yyyy` and `DDth Month YYYY`)
- Sort by date before engineering rolling features — order matters for rolling windows
- With only 3 seasons, early matches in 2020/21 will have fewer previous games available for rolling averages

---
## Comparison: `skysports_match_stats.csv` vs `premier_league_matches.csv`

| Aspect | `premier_league_matches.csv` | `skysports_match_stats.csv` |
|---|---|---|
| Rows | 6,840 | 1,140 |
| Seasons covered | 2000–2020 (20 seasons) | 2020–2023 (3 seasons) |
| Target variable | Broken (H/NH) — needs fixing | Correct (h/d/a) |
| Pre-engineered form features | Yes (streaks, form pts, GD) | No |
| Match statistics | No (only goals) | Yes (possession, shots, tackles etc.) |
| Missing values | None | None |
| Data leakage risk | Low (goals only, easily dropped) | **High** (all stats are post-match) |
| Rolling features needed | Optional (already has form features) | **Required** before use |

---
## Summary of Findings

| # | Finding | Severity | Action Required |
|---|---|---|---|
| 1 | All match stats are post-match (leakage if used directly) | **Critical** | Engineer rolling averages from past matches only |
| 2 | `Goals Home` / `Away Goals` directly reveal the result | **Critical** | Drop before training — never use as features |
| 3 | `class` is the correct 3-class target (h/d/a) | — | Encode as H=2, D=1, A=0 in preprocessing |
| 4 | Inconsistent date formats from scraping | Medium | Parse both formats carefully in preprocessing |
| 5 | `attendance` stored as string with commas | Low | Strip commas, convert to int |
| 6 | Class imbalance — draws underrepresented (22.5%) | Medium | Use balanced class weights in models |
| 7 | Team encoding is anonymous (1–25, no name mapping) | Medium | Accept as-is or recover mapping from scraper |
| 8 | Only 3 seasons of data (1,140 rows) | Medium | Consider combining with `premier_league_matches.csv` |
| 9 | Some possession sums exceed 100 (scraping noise) | Low | Inspect and handle outliers in preprocessing |
| 10 | `links`, `clock` columns not useful | Low | Drop in preprocessing |